"""
Model Training for the Home Credit Default Risk project.

This notebook loads the processed modelling dataset, creates a train-validation
split, trains a Logistic Regression baseline and a LightGBM model, evaluates
both models, and saves trained model artefacts and validation predictions.
"""

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import (
    ID_COLUMN,
    TARGET_COLUMN,
    PROCESSED_DATA_DIR,
    PROCESSED_TRAIN_FILE,
    MODELS_DIR,
    REPORTS_DIR,
    BASELINE_MODEL_FILE,
    TREE_MODEL_FILE,
    FINAL_MODEL_FILE,
)

from src.train_model import (
    split_features_and_target,
    create_train_validation_split,
    train_baseline_model,
    train_lightgbm_model,
    predict_default_probability,
    save_model,
)

from src.evaluate_model import (
    compare_models,
    create_threshold_performance_table,
    create_decile_table,
    create_gain_lift_table,
)

In [3]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

In [4]:
processed_train_path = PROCESSED_DATA_DIR / PROCESSED_TRAIN_FILE

processed_train = pd.read_csv(processed_train_path)

print("Processed training dataset loaded successfully.")
print(f"Processed train shape: {processed_train.shape}")

Processed training dataset loaded successfully.
Processed train shape: (307511, 620)


In [5]:
processed_train.head()

,SK_ID_CURR,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,LANDAREA_AVG,LIVINGAREA_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,LANDAREA_MODE,LIVINGAREA_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,LANDAREA_MEDI,LIVINGAREA_MEDI,NONLIVINGAREA_MEDI,TOTALAREA_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,APP_CREDIT_TO_INCOME_RATIO,APP_ANNUITY_TO_INCOME_RATIO,APP_ANNUITY_TO_CREDIT_RATIO,APP_GOODS_PRICE_TO_CREDIT_RATIO,APP_EMPLOYED_TO_AGE_RATIO,APP_AGE_YEARS,APP_EMPLOYED_YEARS,APP_EMPLOYED_ANOMALY,APP_CHILDREN_TO_FAMILY_RATIO,APP_INCOME_PER_FAMILY_MEMBER,BUREAU_RECORD_COUNT,...,INSTALL_INSTALL_PAYMENT_DIFFERENCE_mean,INSTALL_INSTALL_PAYMENT_DIFFERENCE_max,INSTALL_INSTALL_PAYMENT_DIFFERENCE_min,INSTALL_INSTALL_PAYMENT_DIFFERENCE_sum,INSTALL_INSTALL_PAYMENT_DIFFERENCE_std,INSTALL_INSTALL_PAYMENT_DELAY_DAYS_mean,INSTALL_INSTALL_PAYMENT_DELAY_DAYS_max,INSTALL_INSTALL_PAYMENT_DELAY_DAYS_min,INSTALL_INSTALL_PAYMENT_DELAY_DAYS_sum,INSTALL_INSTALL_PAYMENT_DELAY_DAYS_std,INSTALL_INSTALL_LATE_PAYMENT_FLAG_mean,INSTALL_INSTALL_LATE_PAYMENT_FLAG_max,INSTALL_INSTALL_LATE_PAYMENT_FLAG_min,INSTALL_INSTALL_LATE_PAYMENT_FLAG_sum,INSTALL_INSTALL_LATE_PAYMENT_FLAG_std,NAME_CONTRACT_TYPE_Cash loans,NAME_CONTRACT_TYPE_Revolving loans,CODE_GENDER_F,CODE_GENDER_M,CODE_GENDER_Other,FLAG_OWN_CAR_N,FLAG_OWN_CAR_Y,FLAG_OWN_REALTY_N,FLAG_OWN_REALTY_Y,NAME_TYPE_SUITE_Children,NAME_TYPE_SUITE_Family,NAME_TYPE_SUITE_Other,"NAME_TYPE_SUITE_Spouse, partner",NAME_TYPE_SUITE_Unaccompanied,NAME_INCOME_TYPE_Commercial associate,NAME_INCOME_TYPE_Other,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,NAME_INCOME_TYPE_Working,NAME_EDUCATION_TYPE_Higher education,NAME_EDUCATION_TYPE_Incomplete higher,NAME_EDUCATION_TYPE_Lower secondary,NAME_EDUCATION_TYPE_Other,NAME_EDUCATION_TYPE_Secondary / secondary special,NAME_FAMILY_STATUS_Civil marriage,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Other,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Other,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents,OCCUPATION_TYPE_Accountants,OCCUPATION_TYPE_Cleaning staff,OCCUPATION_TYPE_Cooking staff,OCCUPATION_TYPE_Core staff,OCCUPATION_TYPE_Drivers,OCCUPATION_TYPE_High skill tech staff,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Other,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Unknown,WEEKDAY_APPR_PROCESS_START_FRIDAY,WEEKDAY_APPR_PROCESS_START_MONDAY,WEEKDAY_APPR_PROCESS_START_SATURDAY,WEEKDAY_APPR_PROCESS_START_SUNDAY,WEEKDAY_APPR_PROCESS_START_THURSD

In [6]:
X, y = split_features_and_target(
    df=processed_train,
    target_column=TARGET_COLUMN,
    id_column=ID_COLUMN,
)

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Default rate: {y.mean():.4%}")

Feature matrix shape: (307511, 618)
Target shape: (307511,)
Default rate: 8.0729%


In [7]:
X_train, X_valid, y_train, y_valid = create_train_validation_split(X, y)

print(f"X_train shape: {X_train.shape}")
print(f"X_valid shape: {X_valid.shape}")
print(f"y_train default rate: {y_train.mean():.4%}")
print(f"y_valid default rate: {y_valid.mean():.4%}")

X_train shape: (246008, 618)
X_valid shape: (61503, 618)
y_train default rate: 8.0729%
y_valid default rate: 8.0728%


In [8]:
baseline_model = train_baseline_model(X_train, y_train)

print("Baseline Logistic Regression model trained successfully.")

Baseline Logistic Regression model trained successfully.


In [9]:
lightgbm_model = train_lightgbm_model(X_train, y_train)

print("LightGBM model trained successfully.")

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.121123 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 76757
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 571
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
LightGBM model trained successfully.


In [10]:
baseline_valid_probabilities = predict_default_probability(
    model=baseline_model,
    X=X_valid,
    numeric_only=False,
)

lightgbm_valid_probabilities = predict_default_probability(
    model=lightgbm_model,
    X=X_valid,
    numeric_only=True,
)

print("Validation probabilities generated successfully.")

Validation probabilities generated successfully.


In [11]:
model_comparison = compare_models(
    model_predictions={
        "Logistic Regression Baseline": baseline_valid_probabilities,
        "LightGBM": lightgbm_valid_probabilities,
    },
    y_true=y_valid,
    threshold=0.50,
)

model_comparison

,model_name,roc_auc,gini,ks_statistic,threshold,accuracy,precision,recall,f1_score,true_negatives,false_positives,false_negatives,true_positives
0,LightGBM,0.790005,0.580011,0.442347,0.5,0.757199,0.199687,0.667472,0.307407,43256,13282,1651,3314
1,Logistic Regression Baseline,0.775548,0.551095,0.420092,0.5,0.711640,0.177525,0.707956,0.283868,40253,16285,1450,3515


In [12]:
threshold_performance = create_threshold_performance_table(
    y_true=y_valid,
    y_probability=lightgbm_valid_probabilities,
    thresholds=[0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50],
)

threshold_performance

,threshold,accuracy,precision,recall,f1_score,true_negatives,false_positives,false_negatives,true_positives
0,0.05,0.099491,0.082188,0.998792,0.151879,1160,55378,6,4959
1,0.10,0.177715,0.088494,0.987714,0.162435,6026,50512,61,4904
2,0.15,0.277271,0.098002,0.969386,0.178009,12240,44298,152,4813
3,0.20,0.374242,0.109121,0.942397,0.195594,18338,38200,286,4679
4,0.25,0.460368,0.120869,0.906143,0.213288,23815,32723,466,4499
5,0.30,0.538657,0.134611,0.868479,0.233094,28817,27721,653,4312
6,0.40,0.661756,0.163851,0.777442,0.270659,36840,19698,1105,3860
7,0.50,0.757199,0.199687,0.667472,0.307407,43256,13282,1651,3314


In [13]:
decile_table = create_decile_table(
    y_true=y_valid,
    y_probability=lightgbm_valid_probabilities,
    number_of_bins=10,
)

decile_table

,risk_decile,applicant_count,default_count,average_probability_of_default,actual_default_rate,minimum_probability,maximum_probability,applicant_share,default_share
9,9,6151,1909,0.782041,0.310356,0.693634,0.956014,0.100011,0.384491
8,8,6150,949,0.630589,0.154309,0.571904,0.693621,0.099995,0.191138
7,7,6150,636,0.520031,0.103415,0.470722,0.571863,0.099995,0.128097
6,6,6150,425,0.427414,0.069106,0.386310,0.470707,0.099995,0.085599
5,5,6150,336,0.349215,0.054634,0.313822,0.386306,0.099995,0.067674
4,4,6151,229,0.282563,0.037230,0.253031,0.313802,0.100011,0.046123
3,3,6150,201,0.225262,0.032683,0.198502,0.253023,0.099995,0.040483
2,2,6150,129,0.173777,0.020976,0.149232,0.198495,0.099995,0.025982
1,1,6150,88,0.124654,0.014309,0.100680,0.149217,0.099995,0.017724
0,0,6151,63,0.070097,0.010242,0.005928,0.100675,0.100011,0.012689


In [14]:
gain_lift_table = create_gain_lift_table(
    y_true=y_valid,
    y_probability=lightgbm_valid_probabilities,
    number_of_bins=10,
)

gain_lift_table

,risk_decile,applicant_count,default_count,average_probability_of_default,actual_default_rate,minimum_probability,maximum_probability,applicant_share,default_share,cumulative_applicant_share,cumulative_default_share,lift
0,9,6151,1909,0.782041,0.310356,0.693634,0.956014,0.100011,0.384491,0.100011,0.384491,3.844477
1,8,6150,949,0.630589,0.154309,0.571904,0.693621,0.099995,0.191138,0.200007,0.575629,1.911473
2,7,6150,636,0.520031,0.103415,0.470722,0.571863,0.099995,0.128097,0.300002,0.703726,1.281029
3,6,6150,425,0.427414,0.069106,0.386310,0.470707,0.099995,0.085599,0.399997,0.789325,0.856034
4,5,6150,336,0.349215,0.054634,0.313822,0.386306,0.099995,0.067674,0.499992,0.856999,0.676770
5,4,6151,229,0.282563,0.037230,0.253031,0.313802,0.100011,0.046123,0.600003,0.903122,0.461176
6,3,6150,201,0.225262,0.032683,0.198502,0.253023,0.099995,0.040483,0.699998,0.943605,0.404854
7,2,6150,129,0.173777,0.020976,0.149232,0.198495,0.099995,0.025982,0.799993,0.969587,0.259831
8,1,6150,88,0.124654,0.014309,0.100680,0.149217,0.099995,0.017724,0.899989,0.987311,0.177249
9,0,6151,63,0.070097,0.010242,0.005928,0.100675,0.100011,0.012689,1.000000,1.000000,0.126874


In [15]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

baseline_model_path = save_model(
    model=baseline_model,
    file_name=BASELINE_MODEL_FILE,
)

lightgbm_model_path = save_model(
    model=lightgbm_model,
    file_name=TREE_MODEL_FILE,
)

final_model_path = save_model(
    model=lightgbm_model,
    file_name=FINAL_MODEL_FILE,
)

print(f"Baseline model saved to: {baseline_model_path}")
print(f"LightGBM model saved to: {lightgbm_model_path}")
print(f"Final model saved to: {final_model_path}")

Baseline model saved to: /Users/shriyanshnautiyal/Downloads/credit-risk-modelling-home-credit/models/baseline_logistic_regression.pkl
LightGBM model saved to: /Users/shriyanshnautiyal/Downloads/credit-risk-modelling-home-credit/models/credit_risk_lightgbm.pkl
Final model saved to: /Users/shriyanshnautiyal/Downloads/credit-risk-modelling-home-credit/models/credit_risk_model.pkl


In [16]:
validation_predictions = pd.DataFrame(
    {
        "applicant_id": processed_train.loc[X_valid.index, ID_COLUMN].values
        if ID_COLUMN in processed_train.columns
        else X_valid.index,
        "actual_default": y_valid.values,
        "baseline_probability_of_default": baseline_valid_probabilities.values,
        "probability_of_default": lightgbm_valid_probabilities.values,
    }
)

validation_predictions.head()

,applicant_id,actual_default,baseline_probability_of_default,probability_of_default
0,396899,0,0.277277,0.245891
1,322041,0,0.330339,0.355527
2,220127,0,0.721323,0.846795
3,251531,0,0.385536,0.262195
4,345558,0,0.722593,0.603109


In [17]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

model_comparison.to_csv(
    REPORTS_DIR / "model_comparison.csv",
    index=False,
)

threshold_performance.to_csv(
    REPORTS_DIR / "threshold_performance.csv",
    index=False,
)

decile_table.to_csv(
    REPORTS_DIR / "decile_table.csv",
    index=False,
)

gain_lift_table.to_csv(
    REPORTS_DIR / "gain_lift_table.csv",
    index=False,
)

validation_predictions.to_csv(
    REPORTS_DIR / "validation_predictions.csv",
    index=False,
)

print("Model training outputs saved successfully.")

Model training outputs saved successfully.


In [18]:
print("Training complete.")
print(f"Best model: LightGBM")
print(f"Final model path: {final_model_path}")
print(f"Validation prediction file: {REPORTS_DIR / 'validation_predictions.csv'}")

Training complete.
Best model: LightGBM
Final model path: /Users/shriyanshnautiyal/Downloads/credit-risk-modelling-home-credit/models/credit_risk_model.pkl
Validation prediction file: /Users/shriyanshnautiyal/Downloads/credit-risk-modelling-home-credit/reports/validation_predictions.csv
